# Predictive checklists: full ADT cohort

Primary prediction at ADT initiation; secondary landmarks at days 90/180. Separate platinum and NEPC horizon models, with a resolved-label NEPC sensitivity analysis. Read `README.md` for the censoring estimand, calibration and solver details. This notebook orchestrates the CLI and does not fit models in notebook cells directly.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "survival_common").is_dir())
SUBPROJECT = REPO_ROOT / "COMPASS/survival_analysis/predictive_checklists"
INPUTS_ROOT = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/survival_analysis")
OUTPUT_ROOT = INPUTS_ROOT / "predictive_checklists"
CONFIG = SUBPROJECT / "default_config.json"
LANDMARKS = [0, 90, 180]
HORIZONS = [365, 730, 1095, 1460, 1825]  # 1-5 years
RUN_PREFLIGHT = True
RUN_TRAINING = False
OVERWRITE = False  # archives previous generated results
RESUME = False  # training only: reuse completed horizons of an interrupted run


## Validate cohort inputs

Requires rebuilt endpoint manifests from notebook 01. Preflight validates the full ADT scope, endpoint identity and shared patient splits before any fitting.

In [ ]:
base = [sys.executable, "-m", "COMPASS.survival_analysis.predictive_checklists",
        "--config", str(CONFIG), "--inputs-root", str(INPUTS_ROOT),
        "--landmarks", *map(str, LANDMARKS), "--horizons", *map(str, HORIZONS)]
extra = ["--overwrite"] if OVERWRITE else []
if RUN_PREFLIGHT:
    subprocess.run(base + ["--output-dir", str(OUTPUT_ROOT / "preflight"), "--preflight-only"] + extra,
                   cwd=REPO_ROOT, check=True)


## Train and evaluate

Set `RUN_TRAINING=True` after reviewing preflight counts. The test block remains held out. Runtime scales with checklist lengths, folds, endpoints, landmarks and horizons; begin with landmark 0 and one horizon for a solver pilot.

In [ ]:
RUN_OUTPUT = OUTPUT_ROOT / "full_adt"
if RUN_TRAINING:
    resume = ["--resume"] if RESUME and not OVERWRITE else []
    subprocess.run(base + ["--output-dir", str(RUN_OUTPUT)] + extra + resume,
                   cwd=REPO_ROOT, check=True)


## Inspect results and render figures

Statuses distinguish supported models, inadequate event counts, missing optional comparators and unsupported horizons. These are horizon-specific results; they are not integrated metrics on the existing Cox/XGBoost timeline.

In [ ]:
import pandas as pd

metrics_path = RUN_OUTPUT / "metrics.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics[[c for c in ["endpoint", "landmark_days", "population", "horizon_days",
                                 "model", "status", "auc", "brier", "n_items", "reason"] if c in metrics]])
    subprocess.run(["Rscript", str(SUBPROJECT / "render_figures.R"), str(RUN_OUTPUT)],
                   cwd=REPO_ROOT, check=True)
